# Feature Construction

**Feature Construction** (or Feature Engineering) is the process of creating new features (columns) from your existing data to help a machine learning model uncover hidden patterns. 

Machine Learning models are smart, but they aren't always logical. If you give a model the `Height` and `Width` of a rectangle, a simple linear model might not realize it needs to multiply them together to find the `Area`. As a Data Scientist, you have to do that math for it.

Let's set up a Python sandbox with some raw Real Estate data to see how we can engineer better signals!

In [1]:
import pandas as pd
import numpy as np

# Create a dataset of houses
data = {
    'house_id': [1, 2, 3, 4, 5],
    'price': [250000, 450000, 150000, 800000, 300000],
    'sqft': [1500, 2500, 900, 4000, 1800],
    'bedrooms': [3, 4, 2, 5, 3],
    'year_built': [2005, 2018, 1985, 2022, 1999]
}

df = pd.DataFrame(data)

print("--- Original Raw Data ---")
display(df)

--- Original Raw Data ---


,house_id,price,sqft,bedrooms,year_built
0,1,250000,1500,3,2005
1,2,450000,2500,4,2018
2,3,150000,900,2,1985
3,4,800000,4000,5,2022
4,5,300000,1800,3,1999


# 1. Domain-Specific Features (Ratios)
The most powerful features usually come from **Domain Knowledge**—understanding the business you are working in. 

In real estate, buyers rarely just look at the raw price and raw square footage separately. They look at the **Price per Square Foot**. By combining two raw columns into a new ratio, we give the model a massive shortcut to understanding value.

In [2]:
# Create a copy
df_engineered = df.copy()

# 1. Create a Ratio (Price per Square Foot)
df_engineered['price_per_sqft'] = df_engineered['price'] / df_engineered['sqft']

# 2. Create a Time-based feature (Age of the house)
# Models understand 'Age: 5' much better than trying to subtract '2019' from '2024'
current_year = 2024
df_engineered['house_age_years'] = current_year - df_engineered['year_built']

print("--- Data with Domain Features ---")
display(df_engineered[['house_id', 'price_per_sqft', 'house_age_years']])

--- Data with Domain Features ---


,house_id,price_per_sqft,house_age_years
0,1,166.666667,19
1,2,180.000000,6
2,3,166.666667,39
3,4,200.000000,2
4,5,166.666667,25


# 2. Boolean Flags (Thresholds)
Sometimes, crossing a specific numerical threshold completely changes human behavior. 

For example, a house built in 2020 and a house built in 2022 are both just considered "New Builds." We can help the model by creating a binary (1 or 0) flag that isolates this specific concept.

In [3]:
# Create a flag for "New Construction" (Built in the last 5 years)
# .astype(int) converts the True/False boolean into 1/0
df_engineered['is_new_construction'] = (df_engineered['house_age_years'] <= 5).astype(int)

# Create a flag for "Mansion" (Over 3,500 sqft)
df_engineered['is_mansion'] = (df_engineered['sqft'] > 3500).astype(int)

print("--- Data with Boolean Flags ---")
display(df_engineered[['house_id', 'sqft', 'house_age_years', 'is_new_construction', 'is_mansion']])

--- Data with Boolean Flags ---


,house_id,sqft,house_age_years,is_new_construction,is_mansion
0,1,1500,19,0,0
1,2,2500,6,0,0
2,3,900,39,0,0
3,4,4000,2,1,1
4,5,1800,25,0,0


# 3. Polynomial Features (Capturing Curves)
Standard algorithms (like Linear Regression) assume the world is a straight line: *As Square Footage goes up, Price goes up at a constant rate.*

But what if the relationship is curved? A 1,000 sqft house is cheap. A 3,000 sqft house is expensive. But a 10,000 sqft house isn't just 10x more expensive; it's a mega-mansion, making it 50x more expensive! The price accelerates exponentially. 

To help linear models capture this curve, we construct **Polynomial Features** (squaring or cubing our data).

In [4]:
# Square the Square Footage to capture exponential price growth
df_engineered['sqft_squared'] = df_engineered['sqft'] ** 2

print("--- Capturing Exponential Curves ---")
display(df_engineered[['house_id', 'sqft', 'sqft_squared']])

--- Capturing Exponential Curves ---


,house_id,sqft,sqft_squared
0,1,1500,2250000
1,2,2500,6250000
2,3,900,810000
3,4,4000,16000000
4,5,1800,3240000


# 4. Interaction Terms (Scikit-Learn)
An **Interaction Term** is created when you multiply two completely different features together. This captures the "synergy" between them.

For example, having 5 bedrooms is nice. Being a brand new house is nice. But a *brand new 5-bedroom house* might sell for an massive premium that is greater than just adding the two features together. 

Instead of doing this math by hand, we use **Scikit-Learn's `PolynomialFeatures`**, which automatically generates both squared terms AND interaction terms for us!

In [5]:
from sklearn.preprocessing import PolynomialFeatures

# We want to find interactions between Square Footage and Bedrooms
features_to_interact = df[['sqft', 'bedrooms']]

# 1. Initialize PolynomialFeatures
# degree=2 means it will square things (A^2, B^2) and multiply pairs together (A * B)
# include_bias=False removes an unnecessary column of 1s
poly = PolynomialFeatures(degree=2, include_bias=False)

# 2. Fit and Transform
poly_array = poly.fit_transform(features_to_interact)

# 3. Put it back into a Pandas DataFrame so we can read it
# .get_feature_names_out() automatically names our new math columns!
df_poly = pd.DataFrame(poly_array, columns=poly.get_feature_names_out(['sqft', 'bedrooms']))

print("--- Scikit-Learn Automated Polynomials & Interactions ---")
display(df_poly)

--- Scikit-Learn Automated Polynomials & Interactions ---


,sqft,bedrooms,sqft^2,sqft bedrooms,bedrooms^2
0,1500.0,3.0,2250000.0,4500.0,9.0
1,2500.0,4.0,6250000.0,10000.0,16.0
2,900.0,2.0,810000.0,1800.0,4.0
3,4000.0,5.0,16000000.0,20000.0,25.0
4,1800.0,3.0,3240000.0,5400.0,9.0


*(Look closely at the output! Scikit-Learn kept our original columns (`sqft`, `bedrooms`), created our squared curves (`sqft^2`, `bedrooms^2`), AND multiplied them together to capture their synergy (`sqft * bedrooms`)!)*

---

## Real-World Use Case or Analogy:
Think of Feature Construction like **Baking a Cake**:

* **Raw Data**: You have a bowl of Flour, a bowl of Sugar, and a bowl of raw Eggs. 
* **The Model (Linear Regression)**: If you just hand these raw ingredients to a simple machine learning model, it will try to evaluate the taste of the flour, then the taste of the sugar, and then the taste of the raw eggs independently. The result is gross.
* **Feature Construction**: As the Data Scientist, you act as the Chef. You know that if you mix the Flour and Sugar together, add the Eggs, and apply Heat (creating **Interaction Terms**), the ingredients undergo a chemical reaction to become a Cake.
* **The Result**: You hand the finished Cake to the machine learning model. The model tastes it and says, "Wow, this is incredibly predictive of a delicious dessert!" You didn't add any *new* data from the outside world; you just combined what you already had in a much smarter way.

---